# Đánh giá mô hình `uriel` trên `dataset/val` và ma trận nhầm lẫn

Notebook này sẽ:
- Tải model `models/merged/uriel/w600k_r50.onnx` (InsightFace)
- Trích xuất embedding cho mỗi ảnh trong `dataset/val` (mỗi subfolder là một nhãn)
- Dự đoán nhãn bằng cách lấy ảnh có cosine similarity lớn nhất (không tính chính nó)
- Vẽ ma trận nhầm lẫn (confusion matrix) với seaborn

In [1]:
# Cài đặt thư viện nếu cần
try:
    import insightface
except ImportError:
    !pip install insightface --quiet
    import insightface

try:
    import seaborn as sns
except ImportError:
    !pip install seaborn --quiet
    import seaborn as sns

import os
from glob import glob
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics.pairwise import cosine_similarity

/opt/miniconda3/envs/face_recognition/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Đường dẫn model `uriel` (sửa nếu cần)
uriel_path = 'models/merged/uriel/w600k_r50.onnx'

from insightface.model_zoo import get_model
# Thử load với GPU (ctx_id=0), nếu lỗi thì fallback về CPU (ctx_id=-1)
try:
    uriel = get_model(uriel_path)
    uriel.prepare(ctx_id=0)
    print('Loaded uriel with ctx_id=0 (GPU)')
except Exception as e:
    print('GPU load failed, falling back to CPU:', e)
    uriel = get_model(uriel_path)
    uriel.prepare(ctx_id=-1)
    print('Loaded uriel with ctx_id=-1 (CPU)')

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Loaded uriel with ctx_id=0 (GPU)


/opt/miniconda3/envs/face_recognition/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'CoreMLExecutionProvider, AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [3]:
# Hàm trích xuất 1 embedding từ 1 ảnh
def extract_embedding(img_path, model):
    img = Image.open(img_path).convert('RGB')
    arr = np.asarray(img)
    emb = model.get_feat(arr)
    if isinstance(emb, np.ndarray):
        emb = emb.flatten()
    else:
        emb = np.array(emb).flatten()
    return emb

def extract_embeddings_batch(img_paths, model):
    embs = []
    for p in tqdm(img_paths, desc='Extracting'):
        embs.append(extract_embedding(p, model))
    return np.stack(embs)

In [4]:
# Thu thập danh sách ảnh và nhãn từ dataset/val
val_dir = 'dataset/val'
labels = []
img_paths = []
for label in sorted(os.listdir(val_dir)):
    label_path = os.path.join(val_dir, label)
    if not os.path.isdir(label_path):
        continue
    files = sorted(glob(os.path.join(label_path, '*')))
    for f in files:
        img_paths.append(f)
        labels.append(label)

print(f'Total images: {len(img_paths)}, Total labels: {len(set(labels))}')

Total images: 448, Total labels: 4


In [5]:
# Trích xuất embedding cho toàn bộ val (có thể mất thời gian)
embs = extract_embeddings_batch(img_paths, uriel)

# Tính cosine similarity giữa tất cả ảnh
sims = cosine_similarity(embs, embs)

# Với mỗi ảnh i, loại bỏ chính nó bằng cách set similarity thành -inf, rồi lấy argmax
n = sims.shape[0]
np.fill_diagonal(sims, -np.inf)
pred_idxs = np.argmax(sims, axis=1)
pred_labels = [labels[i] for i in pred_idxs]

# Chuẩn bị ground truth và predicted arrays
y_true = np.array(labels)
y_pred = np.array(pred_labels)

print('Done predictions')

Extracting: 100%|██████████| 448/448 [00:29<00:00, 15.17it/s]

Done predictions


In [ ]:
# Ma trận nhầm lẫn và báo cáo
unique_labels = sorted(list(set(labels)))
cm = confusion_matrix(y_true, y_pred, labels=unique_labels)
cm_df = pd.DataFrame(cm, index=unique_labels, columns=unique_labels)

plt.figure(figsize=(12,10))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion Matrix - uriel on dataset/val')
plt.tight_layout()
plt.show()

print('Classification Report:')
print(classification_report(y_true, y_pred, labels=unique_labels, zero_division=0))

SyntaxError: unterminated string literal (detected at line 14) (3811529668.py, line 14)

## Ghi chú
- Nếu dataset lớn, việc tính ma trận similarity toàn bộ sẽ tốn bộ nhớ; cân nhắc batch hoặc so sánh với centroid mỗi label để giảm chi phí.
- Thay `ctx_id` trong cell load model nếu bạn muốn dùng GPU (`0`) hoặc CPU (`-1`).